<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module12/Lab4.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 4 — Cost + Mixer: Your First Complete QAOA (Instructor)
**Quantum Optimization and Simulation — QAOA Laboratory Series**

Instructor version with answer key.

**Format:** 10–15 minute instructor walkthrough + about 45–60 minutes of independent work.

**Notebook style:** Most code is supplied. Cells marked **YOUR TURN** contain a small value, line, or function for you to complete.

> Qiskit displays measured bitstrings in the order `q_(n-1)...q_0`. When we discuss graph nodes, this notebook often converts them to `q_0...q_(n-1)` using `q0_first(...)`.

## Learning goals
- Assemble the three QAOA stages: superposition, cost, and mixer.
- Observe that the cost layer alone does not change the Z-basis histogram.
- Observe that cost + mixer can increase the probability of high-cut bitstrings.

In [ ]:
# Run this once at the beginning of a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-optimization~=0.7" "qiskit-ibm-runtime~=0.46"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2

SEED = 123
SHOTS = 2048
sampler = SamplerV2(default_shots=SHOTS, seed=SEED)

def run_counts(qc, shots=SHOTS):
    "Run a measured circuit with Aer SamplerV2 and return counts."
    result = sampler.run([qc], shots=shots).result()
    return result[0].data.meas.get_counts()

def q0_first(qiskit_bitstring):
    "Convert Qiskit's displayed q_(n-1)...q_0 bitstring to q_0...q_(n-1)."
    return qiskit_bitstring.replace(" ", "")[::-1]

In [ ]:
def cut_value_q0first(bitstring, edges):
    return sum(bitstring[i] != bitstring[j] for i, j in edges)

def expected_cut_from_counts(counts, edges):
    total = sum(counts.values())
    return sum(
        cut_value_q0first(q0_first(k), edges) * v / total
        for k, v in counts.items()
    )

def apply_cost_layer(qc, edges, gamma):
    for i, j in edges:
        qc.cx(i, j)
        qc.rz(-gamma, j)
        qc.cx(i, j)

def apply_mixer_layer(qc, beta):
    for q in range(qc.num_qubits):
        qc.rx(2 * beta, q)

## Problem: triangle Max-Cut
A triangle has three edges. Any cut can cut at most two of them.

In [ ]:
edges = [(0,1), (1,2), (0,2)]
n = 3

## Part A — H only

In [ ]:
qc_h = QuantumCircuit(n)
qc_h.h(range(n))
qc_h.measure_all()
counts_h = run_counts(qc_h)
print("Expected cut:", expected_cut_from_counts(counts_h, edges))
plot_histogram(counts_h)

## Part B — H + Cost only

In [ ]:
gamma = 3.77

qc_cost = QuantumCircuit(n)
qc_cost.h(range(n))
apply_cost_layer(qc_cost, edges, gamma)
qc_cost.measure_all()

counts_cost = run_counts(qc_cost)
print("Expected cut:", expected_cut_from_counts(counts_cost, edges))
plot_histogram(counts_cost)

**Expected:** the histogram still looks roughly uniform, because the cost operator encodes the scores in phase.

## Part C — H + Cost + Mixer

In [ ]:
beta = 1.885

qc_qaoa = QuantumCircuit(n)
qc_qaoa.h(range(n))
apply_cost_layer(qc_qaoa, edges, gamma)
apply_mixer_layer(qc_qaoa, beta)
qc_qaoa.measure_all()

counts_qaoa = run_counts(qc_qaoa, shots=4096)
print("Expected cut:", expected_cut_from_counts(counts_qaoa, edges))
plot_histogram(counts_qaoa)

**Expected qualitative result:** probability should concentrate strongly on the six bitstrings that cut two edges; `000` and `111` should be suppressed.

### YOUR TURN
Set `beta = 0`. What happens? Then try `beta = 0.4`.

**Question:** What does this experiment say about the role of the mixer?

In [ ]:
test_beta = 0.0   # TODO: after the first run, change to 0.4

qc = QuantumCircuit(n)
qc.h(range(n))
apply_cost_layer(qc, edges, gamma)
apply_mixer_layer(qc, test_beta)
qc.measure_all()

counts = run_counts(qc)
print("Expected cut:", expected_cut_from_counts(counts, edges))
plot_histogram(counts)

## Instructor solution

With `beta = 0`, the mixer is the identity and the Z-basis probabilities remain essentially uniform. With nonzero beta, the mixer lets the phase pattern created by the cost layer interfere and change the probabilities. The exact quality depends on beta.